# 🛰️ SatQuery AI — RS-VLM Fine-Tuning with FMoW 2015

### Fine-tune Qwen2-VL-7B on Satellite VQA & Grounding using 4-bit QLoRA

**Dataset:** FMoW 2015 RGB Dataset

**Selected Classes:**
- Airport
- Crop Field
- Factory / Powerplant
- Lake / Pond
- Port

**Training Dataset:**
- 500 RGB satellite images
- 100 images per class
- Training split
- FMoW 2015

**Model:** Qwen2-VL-7B-Instruct  
**Fine-Tuning:** 4-bit QLoRA  
**GPU:** NVIDIA Tesla T4

---

### Instructions

1. Go to **Runtime → Change runtime type → T4 GPU**.
2. Run the notebook cells sequentially.
3. The FMoW images and dataset files are stored in Google Drive.
4. The next stage will prepare the images into the required **VQA training format**.
5. Fine-tune Qwen2-VL using QLoRA.
6. Save the trained LoRA adapter for integration with **SatQuery AI**.

## 1. Verify GPU Hardware Acceleration

In [1]:
!nvidia-smi

Fri Sep 25 10:22:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies (QLoRA, PEFT, Transformers, BitsAndBytes)

In [2]:
!pip install -q -U "bitsandbytes>=0.46.1" "transformers>=4.45.0" peft accelerate datasets torchvision "pillow>=10.4.0"

print("✅ Dependencies installed.")
print("⚠️ IMPORTANT: If this is your first time running this notebook, restart the Colab session now.")
print("   Go to: Runtime → Restart session")
print("   After restarting, reconnect to the GPU if needed, then continue running the notebook from the next cell.")

✅ Dependencies installed.
⚠️ IMPORTANT: In Google Colab, click 'Runtime' -> 'Restart session' now to reload Pillow before proceeding.


In [13]:
from google.colab import drive
drive.mount('/content/drive')

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


## 3. Prepare FMoW 2015 RGB Dataset

Select 500 training images from 5 classes:
- airport
- crop_field
- factory_or_powerplant
- lake_or_pond
- port

Target: 100 unique training images per class.
The completed dataset will be saved to Google Drive.

In [5]:
# Install AWS CLI for downloading selected FMoW RGB images
!pip -q install awscli

print("✅ AWS CLI installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 40.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
✅ AWS CLI installed.


In [6]:
import os
import json
import subprocess

# FMoW RGB AWS location
S3_ROOT = "s3://spacenet-dataset/Hosted-Datasets/fmow/fmow-rgb"

# Local working folder
LOCAL_ROOT = "/content/fmow_2015"
os.makedirs(LOCAL_ROOT, exist_ok=True)

# Google Drive backup location
DRIVE_ROOT = "/content/drive/MyDrive/SatQuery_AI/FMoW_2015"
os.makedirs(DRIVE_ROOT, exist_ok=True)

print("Local folder:", LOCAL_ROOT)
print("Drive backup:", DRIVE_ROOT)
print("✅ FMoW workspace ready.")

Local folder: /content/fmow_2015
Drive backup: /content/drive/MyDrive/SatQuery_AI/FMoW_2015
✅ FMoW workspace ready.


In [7]:
# Download the FMoW RGB manifest
manifest_path = f"{LOCAL_ROOT}/manifest.json.bz2"

!aws s3 cp --no-sign-request \
    s3://spacenet-dataset/Hosted-Datasets/fmow/fmow-rgb/manifest.json.bz2 \
    "{manifest_path}"

print("✅ FMoW manifest downloaded.")
print("Path:", manifest_path)

download: s3://spacenet-dataset/Hosted-Datasets/fmow/fmow-rgb/manifest.json.bz2 to fmow_2015/manifest.json.bz2
✅ FMoW manifest downloaded.
Path: /content/fmow_2015/manifest.json.bz2


In [8]:
import bz2
import json

manifest_path = "/content/fmow_2015/manifest.json.bz2"

with bz2.open(manifest_path, "rt") as f:
    manifest = json.load(f)

print("Manifest loaded successfully.")
print("Number of entries:", len(manifest))
print("Type:", type(manifest))

Manifest loaded successfully.
Number of entries: 2095393
Type: <class 'list'>


## 4. Prepare FMoW VQA Training Dataset

Create VQA question-answer records from the 500 verified FMoW RGB images.

Each image will be paired with a satellite-image classification question and its FMoW category as the answer.

In [16]:
import json
import os

DRIVE_ROOT = "/content/drive/MyDrive/SatQuery_AI/FMoW_2015"

input_json = os.path.join(
    DRIVE_ROOT,
    "fmow_2015_final_500_training.json"
)

vqa_json = os.path.join(
    DRIVE_ROOT,
    "fmow_2015_vqa_500_training.json"
)

with open(input_json, "r") as f:
    records = json.load(f)

print("Loaded records:", len(records))

vqa_records = []

for item in records:
    image_path = os.path.join(DRIVE_ROOT, item["image"])

    vqa_records.append({
        "image": image_path,
        "category": item["category"],
        "question": "What type of location is shown in this satellite image?",
        "answer": item["category"].replace("_", " ")
    })

with open(vqa_json, "w") as f:
    json.dump(vqa_records, f, indent=2)

print("VQA dataset saved to:", vqa_json)
print("VQA records:", len(vqa_records))

Loaded records: 500
VQA dataset saved to: /content/drive/MyDrive/SatQuery_AI/FMoW_2015/fmow_2015_vqa_500_training.json
VQA records: 500


In [17]:
with open(vqa_json, "r") as f:
    vqa_records = json.load(f)

print("Total VQA records:", len(vqa_records))

categories = {}

for item in vqa_records:
    categories[item["category"]] = categories.get(item["category"], 0) + 1

print("\nCategory counts:")

for category, count in sorted(categories.items()):
    print(f"{category}: {count}")

if len(vqa_records) == 500 and all(count == 100 for count in categories.values()):
    print("\n🎉 VQA DATASET VERIFIED: 500 records, 100 per class")
else:
    print("\n❌ Dataset counts need to be checked.")

Total VQA records: 500

Category counts:
airport: 100
crop_field: 100
factory_or_powerplant: 100
lake_or_pond: 100
port: 100

🎉 VQA DATASET VERIFIED: 500 records, 100 per class


## 5. Load Base VLM in 4-bit NF4 Precision (`Qwen2-VL-7B-Instruct`)

In [18]:
!pip uninstall -y torchaudio

print("✅ Incompatible TorchAudio removed.")

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
✅ Incompatible TorchAudio removed.


In [19]:
import torch
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)

model_id = "Qwen/Qwen2-VL-7B-Instruct"

import bitsandbytes as bnb

print("bitsandbytes version:", bnb.__version__)

processor = AutoProcessor.from_pretrained(model_id)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("✅ Base Qwen2-VL-7B successfully loaded in 4-bit precision!")

bitsandbytes version: 0.50.2


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

✅ Base Qwen2-VL-7B successfully loaded in 4-bit precision!


## 6. Configure LoRA (Parameter-Efficient Fine-Tuning)

In [20]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 8,301,468,160 || trainable%: 0.1216


## 7. Fine-Tuning Execution

In [27]:
import os
import json
import subprocess

DRIVE_ROOT = "/content/drive/MyDrive/SatQuery_AI/FMoW_2015"
input_json = os.path.join(
    DRIVE_ROOT,
    "fmow_2015_final_500_training.json"
)

S3_ROOT = "s3://spacenet-dataset/Hosted-Datasets/fmow/fmow-rgb"

with open(input_json, "r") as f:
    records = json.load(f)

print("Total images to download:", len(records))
print("Starting download...\n")

downloaded = 0
skipped = 0
failed = []

for i, item in enumerate(records, start=1):
    relative_path = item["image"]

    local_path = os.path.join(
        DRIVE_ROOT,
        relative_path
    )

    # Skip if already downloaded
    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        skipped += 1
        print(f"[{i}/500] Already exists: {relative_path}")
        continue

    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    s3_path = f"{S3_ROOT}/{relative_path}"

    result = subprocess.run(
        [
            "aws",
            "s3",
            "cp",
            "--no-sign-request",
            s3_path,
            local_path
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode == 0 and os.path.exists(local_path):
        downloaded += 1
        print(f"[{i}/500] Downloaded: {relative_path}")
    else:
        failed.append(relative_path)
        print(f"[{i}/500] FAILED: {relative_path}")

print("\n================================")
print("DOWNLOAD COMPLETE")
print("================================")
print("Downloaded:", downloaded)
print("Already existed:", skipped)
print("Failed:", len(failed))

if failed:
    print("\nFailed files:")
    for path in failed:
        print(path)
else:
    print("\n🎉 All 500 FMoW images are available.")

Total images to download: 500
Starting download...

[1/500] Downloaded: train/airport/airport_78/airport_78_1_rgb.jpg
[2/500] Downloaded: train/airport/airport_144/airport_144_7_rgb.jpg
[3/500] Downloaded: train/airport/airport_11/airport_11_0_rgb.jpg
[4/500] Downloaded: train/airport/airport_230/airport_230_2_rgb.jpg
[5/500] Downloaded: train/airport/airport_211/airport_211_3_rgb.jpg
[6/500] Downloaded: train/airport/airport_2/airport_2_0_rgb.jpg
[7/500] Downloaded: train/airport/airport_156/airport_156_1_rgb.jpg
[8/500] Downloaded: train/airport/airport_141/airport_141_2_rgb.jpg
[9/500] Downloaded: train/airport/airport_91/airport_91_6_rgb.jpg
[10/500] Downloaded: train/airport/airport_45/airport_45_0_rgb.jpg
[11/500] Downloaded: train/airport/airport_135/airport_135_4_rgb.jpg
[12/500] Downloaded: train/airport/airport_59/airport_59_6_rgb.jpg
[13/500] Downloaded: train/airport/airport_332/airport_332_0_rgb.jpg
[14/500] Downloaded: train/airport/airport_115/airport_115_4_rgb.jpg
[15/5

In [28]:
import os
import json

DRIVE_ROOT = "/content/drive/MyDrive/SatQuery_AI/FMoW_2015"

input_json = os.path.join(
    DRIVE_ROOT,
    "fmow_2015_final_500_training.json"
)

with open(input_json, "r") as f:
    records = json.load(f)

missing = []

for item in records:
    image_path = os.path.join(
        DRIVE_ROOT,
        item["image"]
    )

    if not os.path.isfile(image_path):
        missing.append(item["image"])

print("Total records:", len(records))
print("Images found:", len(records) - len(missing))
print("Images missing:", len(missing))

if missing:
    print("\n❌ Missing images:")
    for path in missing:
        print(path)
else:
    print("\n🎉 ALL 500 IMAGE PATHS VERIFIED SUCCESSFULLY.")

Total records: 500
Images found: 500
Images missing: 0

🎉 ALL 500 IMAGE PATHS VERIFIED SUCCESSFULLY.


In [29]:
import json
import os
from PIL import Image

DRIVE_ROOT = "/content/drive/MyDrive/SatQuery_AI/FMoW_2015"

vqa_json = os.path.join(
    DRIVE_ROOT,
    "fmow_2015_vqa_500_training.json"
)

with open(vqa_json, "r") as f:
    vqa_records = json.load(f)

print("Total VQA records:", len(vqa_records))

# Check that the first image exists
first_image = vqa_records[0]["image"]

print("First image:")
print(first_image)

print("Image exists:", os.path.exists(first_image))

img = Image.open(first_image).convert("RGB")
print("Image size:", img.size)

print("Question:", vqa_records[0]["question"])
print("Answer:", vqa_records[0]["answer"])

Total VQA records: 500
First image:
/content/drive/MyDrive/SatQuery_AI/FMoW_2015/train/airport/airport_78/airport_78_1_rgb.jpg
Image exists: True
Image size: (9636, 5655)
Question: What type of location is shown in this satellite image?
Answer: airport


In [30]:
# Qwen2-VL training collator
# Safely resizes very large FMoW images before loading them into memory.

import torch
from PIL import Image

# Allow PIL to open the very large FMoW images.
Image.MAX_IMAGE_PIXELS = None


class Qwen2VLCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        item = batch[0]   # batch size = 1

        image = Image.open(item["image"]).convert("RGB")

        # Resize extremely large images before Qwen2-VL processing.
        max_side = 2048

        if max(image.size) > max_side:
            scale = max_side / max(image.size)

            new_size = (
                int(image.width * scale),
                int(image.height * scale)
            )

            image = image.resize(new_size)

        question = item["question"]
        answer = item["answer"]

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": question}
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": answer}
                ]
            }
        ]

        prompt_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": question}
                ]
            }
        ]

        full_text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )

        prompt_text = self.processor.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.processor(
            text=[full_text],
            images=[image],
            padding=True,
            return_tensors="pt"
        )

        prompt_inputs = self.processor(
            text=[prompt_text],
            images=[image],
            padding=True,
            return_tensors="pt"
        )

        prompt_length = prompt_inputs["input_ids"].shape[1]

        labels = inputs["input_ids"].clone()

        # Calculate loss only on the assistant answer.
        labels[:, :prompt_length] = -100

        if "attention_mask" in inputs:
            labels[inputs["attention_mask"] == 0] = -100

        inputs["labels"] = labels

        return inputs


collator = Qwen2VLCollator(processor)

print("✅ Updated Qwen2-VL collator created.")
print("✅ Large FMoW images will be safely resized before processing.")

✅ Updated Qwen2-VL collator created.
✅ Large FMoW images will be safely resized before processing.


In [33]:
# Reduce image resolution for T4 GPU memory

processor.image_processor.min_pixels = 256 * 28 * 28
processor.image_processor.max_pixels = 512 * 28 * 28

print("Minimum pixels:", processor.image_processor.min_pixels)
print("Maximum pixels:", processor.image_processor.max_pixels)
print("✅ Qwen2-VL image resolution limited for T4.")

Minimum pixels: 200704
Maximum pixels: 401408
✅ Qwen2-VL image resolution limited for T4.


In [35]:
# Test large FMoW image after reducing Qwen2-VL image resolution

import torch

large_item = None

for item in vqa_records:
    with Image.open(item["image"]) as img:
        if img.width * img.height > 100_000_000:
            large_item = item
            break

if large_item is None:
    print("No image larger than 100 million pixels was found.")
    large_item = vqa_records[0]

print("Test image:")
print(large_item["image"])

with Image.open(large_item["image"]) as img:
    print("Original size:", img.size)
    print("Original pixels:", img.width * img.height)

test_batch = collator([large_item])

print("\nProcessed tensors:")

for key, value in test_batch.items():
    if hasattr(value, "shape"):
        print(f"{key}: {value.shape}")

print("\nInput token count:", test_batch["input_ids"].shape[1])
print("Image patch values:", test_batch["pixel_values"].shape[0])

torch.cuda.empty_cache()

test_batch = {
    key: value.to(model.device) if torch.is_tensor(value) else value
    for key, value in test_batch.items()
}

with torch.no_grad():
    outputs = model(**test_batch)

print("\n==============================")
print("LARGE IMAGE TEST")
print("==============================")
print("Loss:", outputs.loss.item())

if torch.isfinite(outputs.loss):
    print("✅ Large-image test successful.")
else:
    print("❌ Loss is not finite.")

Test image:
/content/drive/MyDrive/SatQuery_AI/FMoW_2015/train/airport/airport_144/airport_144_7_rgb.jpg
Original size: (14452, 14036)
Original pixels: 202848272

Processed tensors:
input_ids: torch.Size([1, 519])
attention_mask: torch.Size([1, 519])
mm_token_type_ids: torch.Size([1, 519])
pixel_values: torch.Size([1936, 1176])
image_grid_thw: torch.Size([1, 3])
labels: torch.Size([1, 519])

Input token count: 519
Image patch values: 1936

LARGE IMAGE TEST
Loss: 4.790796279907227
✅ Large-image test successful.


In [36]:
# Enable gradient checkpointing to reduce GPU memory usage during training

model.config.use_cache = False

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

print("Gradient checkpointing enabled.")
print("Model cache disabled.")
print("✅ Model ready for training.")

Gradient checkpointing enabled.
Model cache disabled.
✅ Model ready for training.


In [38]:
# Training configuration

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/satquery_rsvlm_lora",

    # Training
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,

    # Learning rate
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,

    # Precision
    fp16=True,
    bf16=False,

    # Optimizer
    optim="paged_adamw_8bit",

    # Logging and saving
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=1,

    # Memory
    gradient_checkpointing=True,

    # Custom multimodal collator
    remove_unused_columns=False,

    # Other
    report_to="none",
    dataloader_num_workers=0
)

print("================================")
print("TRAINING CONFIGURATION")
print("================================")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("Optimizer:", training_args.optim)
print("Scheduler:", training_args.lr_scheduler_type)
print("Precision: FP16")
print("✅ Training configuration ready.")

TRAINING CONFIGURATION
Epochs: 3
Batch size: 1
Gradient accumulation: 4
Learning rate: 0.0002
Optimizer: OptimizerNames.PAGED_ADAMW_8BIT
Scheduler: SchedulerType.COSINE
Precision: FP16
✅ Training configuration ready.


In [39]:
# Create the Trainer for the clean training run

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=vqa_records,
    data_collator=collator,
    processing_class=processor
)

print("================================")
print("TRAINER CREATED")
print("================================")
print("Training samples:", len(vqa_records))
print("Epochs:", training_args.num_train_epochs)
print(
    "Effective batch size:",
    training_args.per_device_train_batch_size *
    training_args.gradient_accumulation_steps
)
print("Total optimizer steps:", 375)
print("✅ Trainer is ready for the clean training run.")

TRAINER CREATED
Training samples: 500
Epochs: 3
Effective batch size: 4
Total optimizer steps: 375
✅ Trainer is ready for the clean training run.


In [ ]:
# Start the clean QLoRA fine-tuning run

print("================================")
print("STARTING SATQUERY AI TRAINING")
print("================================")
print("Training samples: 500")
print("Epochs: 3")
print("Total optimizer steps: 375")
print("Batch size: 1")
print("Gradient accumulation: 4")
print("================================")

train_result = trainer.train()

print("\n================================")
print("TRAINING COMPLETE")
print("================================")
print("Training finished successfully.")
print("Final training metrics:")
print(train_result.metrics)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


STARTING SATQUERY AI TRAINING
Training samples: 500
Epochs: 3
Total optimizer steps: 375
Batch size: 1
Gradient accumulation: 4


[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,4.462705
10,1.795415
15,0.713683
20,0.323165
25,0.137323
30,0.142993
35,0.218934
40,0.059294
45,0.041476
50,0.271030


Step,Training Loss
5,4.462705
10,1.795415
15,0.713683
20,0.323165
25,0.137323
30,0.142993
35,0.218934
40,0.059294
45,0.041476
50,0.271030


In [43]:
import os
import shutil

source = "/content/satquery_rsvlm_lora/checkpoint-375"
destination = "/content/drive/MyDrive/SatQuery_AI/checkpoint-375"

# Make sure source exists
if not os.path.exists(source):
    raise FileNotFoundError(f"Checkpoint not found: {source}")

# Create destination parent folder
os.makedirs(
    "/content/drive/MyDrive/SatQuery_AI",
    exist_ok=True
)

# Remove an old copy only if it already exists
if os.path.exists(destination):
    shutil.rmtree(destination)

# Copy the complete checkpoint
shutil.copytree(source, destination)

print("✅ checkpoint-375 copied to Google Drive.")
print("\nSaved at:")
print(destination)

print("\nFiles:")
for file in os.listdir(destination):
    print(" -", file)

✅ checkpoint-375 copied to Google Drive.

Saved at:
/content/drive/MyDrive/SatQuery_AI/checkpoint-375

Files:
 - training_args.bin
 - adapter_config.json
 - processor_config.json
 - README.md
 - adapter_model.safetensors
 - optimizer.pt
 - scheduler.pt
 - trainer_state.json
 - scaler.pt
 - tokenizer_config.json
 - tokenizer.json
 - rng_state.pth
 - chat_template.jinja


## 8. Save and Export LoRA Adapter

In [44]:
# Save final LoRA adapter to Google Drive

import os
import shutil

final_adapter_dir = "/content/drive/MyDrive/SatQuery_AI/satquery_rsvlm_lora"

# Remove old adapter export if it exists
if os.path.exists(final_adapter_dir):
    shutil.rmtree(final_adapter_dir)

# Save trained LoRA adapter
model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)

print("================================")
print("FINAL LORA ADAPTER SAVED")
print("================================")
print("Location:")
print(final_adapter_dir)

print("\nFiles:")

for file in os.listdir(final_adapter_dir):
    print(" -", file)

print("\n✅ LoRA adapter saved to Google Drive.")

FINAL LORA ADAPTER SAVED
Location:
/content/drive/MyDrive/SatQuery_AI/satquery_rsvlm_lora

Files:
 - README.md
 - adapter_model.safetensors
 - adapter_config.json
 - chat_template.jinja
 - tokenizer_config.json
 - tokenizer.json
 - processor_config.json

✅ LoRA adapter saved to Google Drive.


In [45]:
import os

adapter_dir = "/content/drive/MyDrive/SatQuery_AI/satquery_rsvlm_lora"

print("================================")
print("FINAL ADAPTER VERIFICATION")
print("================================")

required_files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "processor_config.json"
]

all_found = True

for file in required_files:
    path = os.path.join(adapter_dir, file)

    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {file} ({size_mb:.2f} MB)")
    else:
        print(f"❌ {file} MISSING")
        all_found = False

print("================================")

if all_found:
    print("🎉 FINAL LORA ADAPTER VERIFIED!")
else:
    print("❌ Verification failed.")

FINAL ADAPTER VERIFICATION
✅ adapter_model.safetensors (38.53 MB)
✅ adapter_config.json (0.00 MB)
✅ processor_config.json (0.00 MB)
🎉 FINAL LORA ADAPTER VERIFIED!


In [47]:
# Correctly test-load the saved LoRA adapter
# using the underlying base model, not an already-wrapped PEFT model.

from peft import PeftModel

adapter_dir = "/content/drive/MyDrive/SatQuery_AI/satquery_rsvlm_lora"

print("================================")
print("CORRECT SAVED ADAPTER TEST")
print("================================")

# Get the original Qwen2-VL base model underneath the trained LoRA wrapper
base_model = model.get_base_model()

print("✅ Underlying base model obtained.")

# Load the saved LoRA adapter onto the base model
test_model = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    is_trainable=False
)

print("\n================================")
print("RESULT")
print("================================")
print("✅ Saved LoRA adapter loaded onto the base model successfully!")
print("Model type:", type(test_model))

CORRECT SAVED ADAPTER TEST
✅ Underlying base model obtained.

RESULT
✅ Saved LoRA adapter loaded onto the base model successfully!
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>


In [49]:
# ============================================
# SATQUERY AI — 5-CLASS INFERENCE TEST
# ============================================

import torch
from PIL import Image

inference_model = test_model
inference_model.eval()

target_classes = [
    "airport",
    "crop_field",
    "factory_or_powerplant",
    "lake_or_pond",
    "port"
]

print("================================")
print("5-CLASS INFERENCE TEST")
print("================================")

results = []

for category in target_classes:

    # Find the first image belonging to this class
    test_item = next(
        item for item in vqa_records
        if item["category"] == category
    )

    image_path = test_item["image"]
    question = test_item["question"]
    expected_answer = test_item["answer"]

    print(f"\nTesting: {category}")
    print("Image:", image_path)

    # Load image
    image = Image.open(image_path).convert("RGB")

    # Create prompt
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Process image + prompt
    inputs = processor(
        text=[prompt],
        images=[image],
        padding=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(inference_model.device)
        if torch.is_tensor(value) else value
        for key, value in inputs.items()
    }

    # Generate prediction
    with torch.no_grad():
        generated_ids = inference_model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    # Remove input tokens
    input_length = inputs["input_ids"].shape[1]

    generated_answer = processor.batch_decode(
        generated_ids[:, input_length:],
        skip_special_tokens=True
    )[0].strip()

    results.append({
        "class": category,
        "expected": expected_answer,
        "predicted": generated_answer
    })

    print("Expected:", expected_answer)
    print("Predicted:", generated_answer)

print("\n================================")
print("5-CLASS TEST RESULTS")
print("================================")

correct = 0

for result in results:
    is_correct = result["expected"].lower() in result["predicted"].lower()

    if is_correct:
        correct += 1

    status = "✅" if is_correct else "❌"

    print(
        f"{status} {result['class']} | "
        f"Expected: {result['expected']} | "
        f"Predicted: {result['predicted']}"
    )

print("\n================================")
print(f"RESULT: {correct}/5 correct")
print("================================")

5-CLASS INFERENCE TEST

Testing: airport
Image: /content/drive/MyDrive/SatQuery_AI/FMoW_2015/train/airport/airport_78/airport_78_1_rgb.jpg
Expected: airport
Predicted: airport

Testing: crop_field
Image: /content/drive/MyDrive/SatQuery_AI/FMoW_2015/train/crop_field/crop_field_6516/crop_field_6516_1_rgb.jpg
Expected: crop field
Predicted: crop field

Testing: factory_or_powerplant
Image: /content/drive/MyDrive/SatQuery_AI/FMoW_2015/train/factory_or_powerplant/factory_or_powerplant_572/factory_or_powerplant_572_2_rgb.jpg
Expected: factory or powerplant
Predicted: factory or powerplant

Testing: lake_or_pond
Image: /content/drive/MyDrive/SatQuery_AI/FMoW_2015/train/lake_or_pond/lake_or_pond_32/lake_or_pond_32_1_rgb.jpg
Expected: lake or pond
Predicted: lake or pond

Testing: port
Image: /content/drive/MyDrive/SatQuery_AI/FMoW_2015/train/port/port_370/port_370_3_rgb.jpg
Expected: port
Predicted: port

5-CLASS TEST RESULTS
✅ airport | Expected: airport | Predicted: airport
✅ crop_field | Ex

In [55]:
import os
import shutil

adapter_dir = "/content/drive/MyDrive/SatQuery_AI/satquery_rsvlm_lora"
zip_base = "/content/drive/MyDrive/SatQuery_AI/satquery_rsvlm_lora"
zip_path = zip_base + ".zip"

if os.path.exists(zip_path):
    print("✅ LoRA adapter ZIP already exists.")
    print("ZIP path:", zip_path)
else:
    zip_path = shutil.make_archive(
        zip_base,
        "zip",
        adapter_dir
    )
    print("✅ LoRA adapter ZIP created successfully.")
    print("ZIP path:", zip_path)

print("ZIP exists:", os.path.exists(zip_path))

✅ LoRA adapter ZIP already exists.
ZIP path: /content/drive/MyDrive/SatQuery_AI/satquery_rsvlm_lora.zip
ZIP exists: True
